# RHINO Thesis Figures — v1
## All datasets from v7 notebook | S11 corrections applied
### Mbatshi Jerry Junior Mbulawa | Jodrell Bank Observatory

---

**Datasets used:**
- `Jodrell_Discone_v2/LNA` — Discone + ZKL-2+ LNA (19 May 2026)
- `Jodrell_Discone_v2/No_LNA` — Discone, no LNA (19 May 2026)
- `Jodrell_Yagi_v2` — Yagi + CobraX LNA (19–20 May 2026)
- `Jodrell_Load_v2` — 50Ω Load + ZKL-2+ LNA, 19800 spectra (~3 hours, 20 May 2026)

**S11 corrections applied:**
- Discone: `discone_jerry_measurement.hd5f` (249 pts in 60–85 MHz)
- Yagi: `yaggi_55_85MHz.hd5f` (332 pts in 60–85 MHz)
- Load: no S11 correction (termination, not antenna)

**Run all cells top to bottom. Do not skip cells.**

In [8]:
# ================================================================
# CELL 1 — Imports
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from scipy.interpolate import UnivariateSpline
import h5py, os, glob
warnings_imported = True
try:
    import warnings; warnings.filterwarnings('ignore')
except: pass

print('NumPy:', np.__version__)
print('PASS Cell 1 — imports OK')

NumPy: 2.3.4
PASS Cell 1 — imports OK


In [9]:
# ================================================================
# CELL 2 — Paths and constants
# ================================================================
BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

# ── v2 dataset paths ─────────────────────────────────────────────
P_LNA    = BASE + '/Jodrell_Discone_v2/LNA'
P_NOLNA  = BASE + '/Jodrell_Discone_v2/No_LNA'
P_YAGI   = BASE + '/Jodrell_Yagi_v2'
P_LOAD   = BASE + '/Jodrell_Load_v2'

# ── S11 files ─────────────────────────────────────────────────────
S11_DISCONE = BASE + '/discone_jerry_measurement.hd5f'
S11_YAGI    = BASE + '/yaggi_55_85MHz.hd5f'
# NOTE: Place the .hd5f files in the Data folder, or update paths above.

# ── Output directory ─────────────────────────────────────────────
OUT = BASE + '/rhino_thesis_figs_final'
os.makedirs(OUT, exist_ok=True)

# ── Hardware constants ────────────────────────────────────────────
FS_MHZ        = 4423.680
N_FFT_COARSE  = 16384
N_FFT_HIRES   = 1048576
N_TAPS        = 4
DF_COARSE_KHZ = FS_MHZ * 1e3 / N_FFT_COARSE
DF_HIRES_KHZ  = FS_MHZ * 1e3 / N_FFT_HIRES

# ── Science band ─────────────────────────────────────────────────
RHINO_LO = 60.0
RHINO_HI = 85.0
REF_LO   = 60.0   # radiometer reference sub-band
REF_HI   = 75.0

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8', 'axes.grid': True,
    'grid.alpha': 0.4, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'lines.linewidth': 1.2,
})
COL = {'fft':'#1f77b4', 'pfb':'#d62728', 'lna':'#ff7f0e',
       'nolna':'#2ca02c', 'yagi':'#9467bd', 'load':'#17becf'}

print('Output directory:', OUT)
print('PASS Cell 2')

Output directory: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_thesis_figs_final
PASS Cell 2


In [10]:
# ================================================================
# CELL 3 — S11 correction functions
# Jordan's method: fit cubic spline to |S11| in dB,
# compute mismatch efficiency eta = 1 - |Gamma|^2,
# apply correction: spec_dB_corrected = spec_dB - 10*log10(eta)
# ================================================================
import h5py
from scipy.interpolate import UnivariateSpline

def load_s11(path, name=''):
    """Load S11 file and return (freq_mhz, s11_db) arrays."""
    with h5py.File(path, 'r') as f:
        freq_hz = f['Frequencies'][()].astype(np.float64)
        s11_cplx = f['s11'][()].astype(np.complex128)
    freq_mhz = freq_hz / 1e6
    s11_db   = 20 * np.log10(np.maximum(np.abs(s11_cplx), 1e-10))
    print('  %s S11: %d pts, %.1f-%.1f MHz' % (
          name, len(freq_mhz), freq_mhz.min(), freq_mhz.max()))
    return freq_mhz, s11_db

def build_s11_correction(freq_mhz, s11_db, target_freq_mhz):
    """Fit spline to |S11| dB and return correction array in dB.
    Correction = -10*log10(eta) where eta = 1 - |Gamma|^2.
    ADD this to the raw spectrum to correct for mismatch."""
    spline  = UnivariateSpline(freq_mhz, s11_db, k=3, s=0.5)
    s11_fit = spline(target_freq_mhz)
    gamma   = 10**(s11_fit / 20.0)
    eta     = 1.0 - gamma**2
    eta     = np.clip(eta, 0.01, 1.0)  # avoid log(0)
    corr_db = -10 * np.log10(eta)
    return corr_db, s11_fit, eta

# Load both S11 files
print('Loading S11 measurements...')
try:
    freq_s11_discone, s11_db_discone = load_s11(S11_DISCONE, 'Discone')
    HAS_DISCONE_S11 = True
except Exception as e:
    print('  WARN: discone S11 not found —', e); HAS_DISCONE_S11 = False
try:
    freq_s11_yagi, s11_db_yagi = load_s11(S11_YAGI, 'Yagi')
    HAS_YAGI_S11 = True
except Exception as e:
    print('  WARN: Yagi S11 not found —', e); HAS_YAGI_S11 = False

print('PASS Cell 3 — S11 functions ready')

Loading S11 measurements...
  Discone S11: 399 pts, 50.0-90.0 MHz
  Yagi S11: 399 pts, 55.0-85.0 MHz
PASS Cell 3 — S11 functions ready


In [11]:
# ================================================================
# CELL 4 — Loader utilities and frequency axes
# ================================================================
def load_npy(folder, fname):
    path = os.path.join(folder, fname)
    if os.path.exists(path):
        return np.load(path, allow_pickle=False)
    print('  MISS:', fname)
    return None

def _parse_N(fp):
    return int(fp.split('_N')[-1].replace('.npy',''))

def glob_snaps(folder, pattern):
    files = glob.glob(os.path.join(folder, pattern))
    if not files: return [], []
    # CRITICAL: sort NUMERICALLY by N, not lexicographically.
    # glob/sorted() order strings, so "N108" < "N12" < "N96" — which made the
    # two-point radiometer ratio use adjacent N values as its endpoints.
    files = sorted(files, key=_parse_N)
    Ns, arrs = [], []
    for fp in files:
        try:
            Ns.append(_parse_N(fp))
            arrs.append(np.load(fp, allow_pickle=False))
        except: pass
    return Ns, arrs

def freq_to_z(f_mhz):
    return 1420.405751786 / f_mhz - 1

# Frequency axes
freq_coarse = np.fft.rfftfreq(N_FFT_COARSE, d=1.0/FS_MHZ)
freq_hires  = np.fft.rfftfreq(N_FFT_HIRES,  d=1.0/FS_MHZ)

def band_idx(freq, lo, hi):
    return np.searchsorted(freq, lo), np.searchsorted(freq, hi)+1

lo_c, hi_c = band_idx(freq_coarse, RHINO_LO, RHINO_HI)
lo_h, hi_h = band_idx(freq_hires,  RHINO_LO, RHINO_HI)

print('Coarse bins in 60-85 MHz: %d (%.1f kHz/bin)' % (hi_c-lo_c, DF_COARSE_KHZ))
print('Hires  bins in 60-85 MHz: %d (%.3f kHz/bin)' % (hi_h-lo_h, DF_HIRES_KHZ))
print('PASS Cell 4 — snapshots now sorted NUMERICALLY by N')


Coarse bins in 60-85 MHz: 93 (270.0 kHz/bin)
Hires  bins in 60-85 MHz: 5927 (4.219 kHz/bin)
PASS Cell 4 — snapshots now sorted NUMERICALLY by N


In [12]:
# ================================================================
# CELL 5 — Load all v2 datasets
# ================================================================
print('Loading Discone + LNA...')
ds_lna = {
    'freq_c'  : load_npy(P_LNA, 'freq_coarse_20260519_180650.npy'),
    'freq_h'  : load_npy(P_LNA, 'freq_hires_20260519_190811.npy'),
    'fft_c'   : load_npy(P_LNA, 'fft_coarse_20260519_180650.npy'),
    'pfb_c'   : load_npy(P_LNA, 'pfb_coarse_20260519_180650.npy'),
    'fft_h'   : load_npy(P_LNA, 'fft_hires_20260519_190811.npy'),
    'integ_f' : load_npy(P_LNA, 'integ_final_20260519_191513.npy'),
    'integ_freq': load_npy(P_LNA, 'integ_freq_20260519_191513.npy'),
    'wf_fft'  : load_npy(P_LNA, 'wf_fft_20260519_180713.npy'),
    'wf_pfb'  : load_npy(P_LNA, 'wf_pfb_20260519_180713.npy'),
    'wf_fft_q': load_npy(P_LNA, 'wf_fft_quiet_20260519_180713.npy'),
    'wf_pfb_q': load_npy(P_LNA, 'wf_pfb_quiet_20260519_180713.npy'),
    'wf_fft_r': load_npy(P_LNA, 'wf_fft_rfi_20260519_180713.npy'),
    'wf_pfb_r': load_npy(P_LNA, 'wf_pfb_rfi_20260519_180713.npy'),
    'wf_hires': load_npy(P_LNA, 'wf_hires_20260519_190818.npy'),
    'wf_times': load_npy(P_LNA, 'wf_times_20260519_180713.npy'),
    'wf_rms'  : load_npy(P_LNA, 'wf_rms_20260519_180713.npy'),
    'raw_ts'  : load_npy(P_LNA, 'raw_timestream_20260519_190806.npy'),
    'label'   : 'Discone + ZKL-2+ LNA',
}
Ns_lna, snaps_lna = glob_snaps(P_LNA, 'integ_snap_20260519_191513_N*.npy')
ds_lna['snap_Ns'] = Ns_lna; ds_lna['snaps'] = snaps_lna
print('  Waterfall shape:', ds_lna['wf_fft'].shape if ds_lna['wf_fft'] is not None else 'MISS')
print('  Integration snaps:', len(snaps_lna))

print('Loading Discone No LNA...')
ds_nolna = {
    'freq_c'  : load_npy(P_NOLNA, 'freq_coarse_20260519_192441.npy'),
    'freq_h'  : load_npy(P_NOLNA, 'freq_hires_20260519_202523.npy'),
    'fft_c'   : load_npy(P_NOLNA, 'fft_coarse_20260519_192441.npy'),
    'pfb_c'   : load_npy(P_NOLNA, 'pfb_coarse_20260519_192441.npy'),
    'fft_h'   : load_npy(P_NOLNA, 'fft_hires_20260519_202523.npy'),
    'integ_f' : load_npy(P_NOLNA, 'integ_final_20260519_203216.npy'),
    'integ_freq': load_npy(P_NOLNA, 'integ_freq_20260519_203216.npy'),
    'wf_fft'  : load_npy(P_NOLNA, 'wf_fft_20260519_192451.npy'),
    'wf_pfb'  : load_npy(P_NOLNA, 'wf_pfb_20260519_192451.npy'),
    'wf_fft_q': load_npy(P_NOLNA, 'wf_fft_quiet_20260519_192451.npy'),
    'wf_pfb_q': load_npy(P_NOLNA, 'wf_pfb_quiet_20260519_192451.npy'),
    'wf_fft_r': load_npy(P_NOLNA, 'wf_fft_rfi_20260519_192451.npy'),
    'wf_pfb_r': load_npy(P_NOLNA, 'wf_pfb_rfi_20260519_192451.npy'),
    'wf_hires': load_npy(P_NOLNA, 'wf_hires_20260519_202527.npy'),
    'wf_times': load_npy(P_NOLNA, 'wf_times_20260519_192451.npy'),
    'wf_rms'  : load_npy(P_NOLNA, 'wf_rms_20260519_192451.npy'),
    'raw_ts'  : load_npy(P_NOLNA, 'raw_timestream_20260519_202519.npy'),
    'label'   : 'Discone, no LNA',
}
Ns_nolna, snaps_nolna = glob_snaps(P_NOLNA, 'integ_snap_20260519_203216_N*.npy')
ds_nolna['snap_Ns'] = Ns_nolna; ds_nolna['snaps'] = snaps_nolna
print('  Integration snaps:', len(snaps_nolna))

print('Loading Yagi + CobraX LNA...')
ds_yagi = {
    'freq_c'  : load_npy(P_YAGI, 'freq_coarse_20260519_232707.npy'),
    'freq_h'  : load_npy(P_YAGI, 'freq_hires_20260520_002851.npy'),
    'fft_c'   : load_npy(P_YAGI, 'fft_coarse_20260519_232707.npy'),
    'pfb_c'   : load_npy(P_YAGI, 'pfb_coarse_20260519_232707.npy'),
    'fft_h'   : load_npy(P_YAGI, 'fft_hires_20260520_002851.npy'),
    'integ_f' : load_npy(P_YAGI, 'integ_final_20260520_003622.npy'),
    'integ_freq': load_npy(P_YAGI, 'integ_freq_20260520_003622.npy'),
    'wf_fft'  : load_npy(P_YAGI, 'wf_fft_20260519_232717.npy'),
    'wf_pfb'  : load_npy(P_YAGI, 'wf_pfb_20260519_232717.npy'),
    'wf_fft_q': load_npy(P_YAGI, 'wf_fft_quiet_20260519_232717.npy'),
    'wf_pfb_q': load_npy(P_YAGI, 'wf_pfb_quiet_20260519_232717.npy'),
    'wf_fft_r': load_npy(P_YAGI, 'wf_fft_rfi_20260519_232717.npy'),
    'wf_pfb_r': load_npy(P_YAGI, 'wf_pfb_rfi_20260519_232717.npy'),
    'wf_hires': load_npy(P_YAGI, 'wf_hires_20260520_002900.npy'),
    'wf_times': load_npy(P_YAGI, 'wf_times_20260519_232717.npy'),
    'wf_rms'  : load_npy(P_YAGI, 'wf_rms_20260519_232717.npy'),
    'raw_ts'  : load_npy(P_YAGI, 'raw_timestream_20260520_002846.npy'),
    'label'   : 'Yagi + CobraX LNA',
}
Ns_yagi, snaps_yagi = glob_snaps(P_YAGI, 'integ_snap_20260520_003622_N*.npy')
ds_yagi['snap_Ns'] = Ns_yagi; ds_yagi['snaps'] = snaps_yagi
print('  Integration snaps:', len(snaps_yagi))

print('Loading Load + ZKL-2+ LNA (3-hour)...')
ds_load = {
    'freq_c'  : load_npy(P_LOAD, 'freq_coarse_20260520_002628.npy'),
    'fft_c'   : load_npy(P_LOAD, 'fft_coarse_20260520_002628.npy'),
    'pfb_c'   : load_npy(P_LOAD, 'pfb_coarse_20260520_002628.npy'),
    'integ_f' : load_npy(P_LOAD, 'integ_final_20260520_003037.npy'),
    'integ_freq': load_npy(P_LOAD, 'integ_freq_20260520_003037.npy'),
    'label'   : '50Ohm Load + ZKL-2+ LNA (~3 h)',
}
# Use the main 3-hour integration series (N=100 to N=19800)
Ns_load, snaps_load = glob_snaps(P_LOAD, 'integ_snap_20260520_003037_N*.npy')
ds_load['snap_Ns'] = Ns_load; ds_load['snaps'] = snaps_load
print('  Integration snaps (3h series):', len(snaps_load))
print('  Max N:', max(Ns_load) if Ns_load else 'N/A')
print()
print('PASS Cell 5 — all datasets loaded')

Loading Discone + LNA...
  Waterfall shape: (360, 93)
  Integration snaps: 30
Loading Discone No LNA...
  Integration snaps: 30
Loading Yagi + CobraX LNA...
  Integration snaps: 30
Loading Load + ZKL-2+ LNA (3-hour)...
  Integration snaps (3h series): 198
  Max N: 19800

PASS Cell 5 — all datasets loaded


In [13]:
# ================================================================
# CELL 6 — Build S11 corrections for each dataset
# ================================================================

def apply_s11_to_spectrum(spec_db, freq_mhz_spec, freq_s11, s11_db_raw):
    """Apply S11 correction to a spectrum array.
    Returns corrected spectrum and correction array (both in dB)."""
    corr_db, s11_fit, eta = build_s11_correction(
        freq_s11, s11_db_raw, freq_mhz_spec)
    return spec_db + corr_db, corr_db, eta

print('Building S11 corrections...')

# ── Discone correction (applies to LNA and No_LNA) ───────────────
if HAS_DISCONE_S11:
    f_rhino_c = freq_coarse[lo_c:hi_c]
    corr_discone_c, s11_fit_d, eta_discone_c = build_s11_correction(
        freq_s11_discone, s11_db_discone, f_rhino_c)

    f_rhino_h = freq_hires[lo_h:hi_h]
    corr_discone_h, _, eta_discone_h = build_s11_correction(
        freq_s11_discone, s11_db_discone, f_rhino_h)

    print('Discone coarse correction: %.3f to %.3f dB in 60-85 MHz' %
          (corr_discone_c.min(), corr_discone_c.max()))
    print('Discone mean eta: %.4f (%.2f dB correction on average)' %
          (eta_discone_c.mean(), corr_discone_c.mean()))
else:
    corr_discone_c = np.zeros(hi_c - lo_c)
    corr_discone_h = np.zeros(hi_h - lo_h)
    print('WARN: no discone S11 — zero correction applied')

# ── Yagi correction ──────────────────────────────────────────────
if HAS_YAGI_S11:
    f_rhino_c = freq_coarse[lo_c:hi_c]
    corr_yagi_c, s11_fit_y, eta_yagi_c = build_s11_correction(
        freq_s11_yagi, s11_db_yagi, f_rhino_c)

    f_rhino_h = freq_hires[lo_h:hi_h]
    corr_yagi_h, _, eta_yagi_h = build_s11_correction(
        freq_s11_yagi, s11_db_yagi, f_rhino_h)

    print('Yagi coarse correction: %.3f to %.3f dB in 60-85 MHz' %
          (corr_yagi_c.min(), corr_yagi_c.max()))
    print('Yagi mean eta: %.4f (%.2f dB correction on average)' %
          (eta_yagi_c.mean(), corr_yagi_c.mean()))
else:
    corr_yagi_c = np.zeros(hi_c - lo_c)
    corr_yagi_h = np.zeros(hi_h - lo_h)
    print('WARN: no Yagi S11 — zero correction applied')

print('PASS Cell 6 — S11 corrections built')

Building S11 corrections...
Discone coarse correction: 0.060 to 1.226 dB in 60-85 MHz
Discone mean eta: 0.9289 (0.33 dB correction on average)
Yagi coarse correction: 4.443 to 5.648 dB in 60-85 MHz
Yagi mean eta: 0.3096 (5.10 dB correction on average)
PASS Cell 6 — S11 corrections built


In [14]:
# ================================================================
# CELL 7 — Fig 1: Theoretical filter response (FFT vs PFB)
# No data needed — computed analytically from prototype filter.
# ================================================================
pfb_len     = N_FFT_COARSE * N_TAPS
t_pfb       = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs  = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
hann_window = np.hanning(N_FFT_COARSE).astype(np.float64)

# Frequency response via zero-padding
def channel_response(h, N=N_FFT_COARSE, pad=16):
    h_pad = np.zeros(N * pad, dtype=np.float64)
    h_pad[:len(h)] = h[:N*pad] if len(h) >= N*pad else np.pad(h, (0, N*pad - len(h)))
    H = np.fft.rfft(h_pad)
    H_db = 20 * np.log10(np.maximum(np.abs(H) / np.abs(H).max(), 1e-10))
    f_bins = np.fft.rfftfreq(len(h_pad)) * pad
    return f_bins, H_db

# FFT: response of one channel = Hann window response
f_fft, H_fft = channel_response(hann_window)
# PFB: response of prototype filter first N coefficients
pfb_proto_one = pfb_coeffs[:N_FFT_COARSE]
f_pfb, H_pfb = channel_response(pfb_proto_one)

fig, ax = plt.subplots(figsize=(10, 5))
mask_plot = (f_fft >= -4) & (f_fft <= 4)
ax.plot(f_fft[mask_plot], H_fft[mask_plot],
        color=COL['fft'], lw=1.5, label='FFT (Hann)  first sidelobe ≈ −13 dB')
mask_plot2 = (f_pfb >= -4) & (f_pfb <= 4)
ax.plot(f_pfb[mask_plot2], H_pfb[mask_plot2],
        color=COL['pfb'], lw=1.5,
        label='PFB (Hann-sinc, %d taps)  sidelobe ≈ −57 dB' % N_TAPS)
ax.axhline(-13, color=COL['fft'], lw=0.7, ls=':', alpha=0.6)
ax.axhline(-57, color=COL['pfb'], lw=0.7, ls=':', alpha=0.6)
ax.axvline(-0.5, color='grey', lw=0.7, ls='--', alpha=0.4)
ax.axvline( 0.5, color='grey', lw=0.7, ls='--', alpha=0.4)
ax.set_xlim(-4, 4); ax.set_ylim(-90, 5)
ax.set_xlabel('Frequency offset (bins)')
ax.set_ylabel('Channel response (dB)')
ax.set_title('Fig 1 — Theoretical FFT vs PFB Channel Frequency Response')
ax.legend()
fig.tight_layout()
fig.savefig(OUT + '/fig1_filter_response.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 1 saved')

ValueError: could not broadcast input array from shape (262144,) into shape (16384,)

In [15]:
# ================================================================
# CELL 8 — Fig 3: Wideband spectrum 0-500 MHz
# Shows LNA, No-LNA, Yagi, and Load — raw power (no S11 correction
# on wideband; correction only applied in science band plots).
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

for ax, ds, col_fft, col_pfb, apply_corr, corr in [
    (axes[0], ds_lna,  COL['fft'], COL['pfb'], False, None),
    (axes[1], ds_yagi, COL['fft'], COL['pfb'], False, None),
    (axes[2], ds_load, COL['fft'], COL['pfb'], False, None),
]:
    if ds['fft_c'] is None or ds['freq_c'] is None: continue
    # Use native freq axis from file, convert if stored in THz
    fq = ds['freq_c']
    if fq.max() < 10: fq = fq * 1e6  # stored in THz? convert
    if fq.max() > 1e6: fq = fq / 1e6  # stored in Hz? convert
    mask = fq <= 500
    ax.plot(fq[mask], ds['fft_c'][mask], color=col_fft,
            lw=0.6, alpha=0.85, label='FFT (Hann)')
    if ds['pfb_c'] is not None:
        ax.plot(fq[mask], ds['pfb_c'][mask], color=col_pfb,
                lw=0.6, alpha=0.85, label='PFB (%d taps)' % N_TAPS)
    ax.axvspan(RHINO_LO, RHINO_HI, alpha=0.12, color='gold',
               label='RHINO band (60-85 MHz)')
    ax.axvspan(87.5, 108.0, alpha=0.07, color='red',
               label='FM (87.5-108 MHz)')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_title(ds['label'])
    ax.legend(fontsize=8, ncol=2)
    ax.set_xlim(0, 500)

fig.suptitle('Fig 3 — Wideband Spectrum 0-500 MHz | RHINO Science Band Highlighted',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig3_wideband_spectrum.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 3 saved')

PASS Fig 3 saved


In [16]:
# ================================================================
# CELL 9 — Fig 4: RHINO band zoom 60-85 MHz with S11 correction
# Shows FFT and PFB for each sky dataset with and without S11.
# Dual x-axis: frequency (MHz) and cosmological redshift z.
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(11, 12))

datasets_fig4 = [
    (axes[0], ds_lna,   corr_discone_c, 'Discone + ZKL-2+ LNA'),
    (axes[1], ds_nolna, corr_discone_c, 'Discone, no LNA'),
    (axes[2], ds_yagi,  corr_yagi_c,    'Yagi + CobraX LNA'),
]

for ax, ds, corr, title in datasets_fig4:
    if ds['fft_c'] is None: continue
    fq = freq_coarse
    mask = (fq >= RHINO_LO) & (fq <= RHINO_HI)
    f_band = fq[mask]

    fft_raw = ds['fft_c'][mask]
    pfb_raw = ds['pfb_c'][mask] if ds['pfb_c'] is not None else None

    # S11 correction
    fft_corr = fft_raw + corr
    pfb_corr = pfb_raw + corr if pfb_raw is not None else None

    fft_std = float(np.std(fft_raw))
    pfb_std = float(np.std(pfb_raw)) if pfb_raw is not None else 0
    fft_std_c = float(np.std(fft_corr))

    ax.plot(f_band, fft_raw,  color=COL['fft'], lw=1.0,
            alpha=0.5, ls='--', label='FFT raw')
    ax.plot(f_band, fft_corr, color=COL['fft'], lw=1.2,
            label='FFT + S11 corr  std=%.3f dB' % fft_std_c)
    if pfb_corr is not None:
        ax.plot(f_band, pfb_raw,  color=COL['pfb'], lw=1.0,
                alpha=0.5, ls='--', label='PFB raw')
        ax.plot(f_band, pfb_corr, color=COL['pfb'], lw=1.2,
                label='PFB + S11 corr  std=%.3f dB' % float(np.std(pfb_corr)))

    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO, RHINO_HI)
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)

    # Redshift axis
    ax2 = ax.twiny()
    f_ticks = np.linspace(RHINO_LO, RHINO_HI, 6)
    ax2.set_xlim(RHINO_LO, RHINO_HI)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)

fig.suptitle('Fig 4 — RHINO Science Band 60-85 MHz | FFT vs PFB | S11 Corrected',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig4_rhino_band_zoom.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 4 saved')

ValueError: operands could not be broadcast together with shapes (92,) (93,) 

In [17]:
# ================================================================
# CELL 10 — Fig 5: Waterfall comparison (FFT vs PFB)
# Three sub-band views for LNA and No-LNA datasets.
# Yagi waterfall also generated.
# ================================================================

def make_waterfall_fig(ds, lo, hi, wf_key_f, wf_key_p, label, fname):
    wf_f = ds.get(wf_key_f); wf_p = ds.get(wf_key_p)
    times = ds.get('wf_times'); rms = ds.get('wf_rms')
    if wf_f is None: print('  SKIP %s — data missing' % label); return

    n_fr, n_bins = wf_f.shape
    f_ax = np.linspace(lo, hi, n_bins)
    diff = wf_f - wf_p if wf_p is not None else np.zeros_like(wf_f)

    flat = np.concatenate([wf_f.ravel(),
                           wf_p.ravel() if wf_p is not None else []])
    flat = flat[flat != 0]
    vmin = float(np.percentile(flat,  2)) if len(flat) else 20
    vmax = float(np.percentile(flat, 98)) if len(flat) else 60
    vlim = max(float(np.percentile(np.abs(diff), 99)), 0.5)

    # Time axis
    if times is not None and len(times) == n_fr:
        t_ax = times
    else:
        t_ax = np.arange(n_fr) * 10.0

    # Stripe detection
    if rms is not None and len(rms) == n_fr:
        rms_med = np.median(rms); rms_std = np.std(rms)
        stripes = np.where(rms > rms_med + 2*rms_std)[0]
    else:
        stripes = np.array([])

    ext = [f_ax[0], f_ax[-1], t_ax[-1], 0]

    fig = plt.figure(figsize=(18, 11))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

    for ax, wf, title, cmap, vm1, vm2 in [
        (fig.add_subplot(gs[0,0]), wf_f,  'FFT (Hann)',       'viridis', vmin, vmax),
        (fig.add_subplot(gs[0,1]), wf_p if wf_p is not None else np.zeros_like(wf_f),
                                           'PFB (%d taps)'%N_TAPS, 'viridis', vmin, vmax),
        (fig.add_subplot(gs[0,2]), diff,  'FFT − PFB',        'RdBu_r', -vlim, vlim)]:
        im = ax.imshow(wf, aspect='auto', origin='upper',
                       extent=ext, cmap=cmap, vmin=vm1, vmax=vm2,
                       interpolation='nearest')
        for sf in stripes:
            if sf < n_fr: ax.axhline(t_ax[int(sf)], color='yellow', lw=0.4, alpha=0.6)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Time (s)')
        ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, label='dB')

    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(f_ax, np.mean(wf_f, axis=0), color=COL['fft'], lw=1.2, label='FFT mean')
    if wf_p is not None:
        ax4.plot(f_ax, np.mean(wf_p, axis=0), color=COL['pfb'], lw=1.2, label='PFB mean')
    mean_diff = float(np.mean(np.abs(diff)))
    ax4.set_xlabel('Frequency (MHz)')
    ax4.set_ylabel('Mean power (dB, arb.)')
    ax4.set_title('%d frames | mean|FFT-PFB|=%.4f dB | Stripes: %d/%d' %
                  (n_fr, mean_diff, len(stripes), n_fr))
    ax4.set_xlim(lo, hi); ax4.legend()
    fig.suptitle('FFT vs PFB Waterfall | %.0f-%.0f MHz | %s | v7 normalisation' %
                 (lo, hi, label), fontsize=10)
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('  Saved:', os.path.basename(fname))
    print('  Stripes: %d/%d | mean|FFT-PFB|=%.4f dB' % (len(stripes), n_fr, mean_diff))

print('Generating Fig 5 waterfalls...')
for ds, corr, tag, lbl in [
    (ds_lna,   corr_discone_c, 'lna',   'Discone+LNA'),
    (ds_nolna, corr_discone_c, 'nolna', 'Discone No-LNA'),
    (ds_yagi,  corr_yagi_c,   'yagi',  'Yagi+CobraX LNA'),
]:
    for wkf, wkp, lo, hi, sub in [
        ('wf_fft',  'wf_pfb',  RHINO_LO, RHINO_HI, 'full'),
        ('wf_fft_q','wf_pfb_q', 65.0,    78.0,      'quiet'),
        ('wf_fft_r','wf_pfb_r', 60.0,    70.0,      'rfi'),
    ]:
        if ds.get(wkf) is None: continue
        make_waterfall_fig(
            ds, lo, hi, wkf, wkp,
            '%s %s' % (lbl, sub.title()),
            OUT + '/fig5_waterfall_%s_%s.png' % (tag, sub)
        )

print('PASS Cell 10 — Fig 5 waterfalls complete')

Generating Fig 5 waterfalls...
  Saved: fig5_waterfall_lna_full.png
  Stripes: 12/360 | mean|FFT-PFB|=8.0084 dB
  Saved: fig5_waterfall_lna_quiet.png
  Stripes: 12/360 | mean|FFT-PFB|=7.5834 dB
  Saved: fig5_waterfall_lna_rfi.png
  Stripes: 12/360 | mean|FFT-PFB|=8.9144 dB
  Saved: fig5_waterfall_nolna_full.png
  Stripes: 14/360 | mean|FFT-PFB|=7.9602 dB
  Saved: fig5_waterfall_nolna_quiet.png
  Stripes: 14/360 | mean|FFT-PFB|=8.0209 dB
  Saved: fig5_waterfall_nolna_rfi.png
  Stripes: 14/360 | mean|FFT-PFB|=8.0943 dB
  Saved: fig5_waterfall_yagi_full.png
  Stripes: 10/360 | mean|FFT-PFB|=7.4955 dB
  Saved: fig5_waterfall_yagi_quiet.png
  Stripes: 10/360 | mean|FFT-PFB|=7.5950 dB
  Saved: fig5_waterfall_yagi_rfi.png
  Stripes: 10/360 | mean|FFT-PFB|=7.5516 dB
PASS Cell 10 — Fig 5 waterfalls complete


In [18]:
# ================================================================
# CELL 11 — Fig 6: Hi-res spectrum at 4.219 kHz/bin
# Shows LNA (discone), No-LNA (discone), and Yagi.
# S11 correction applied to the science band portion.
# Dual x-axis: frequency (MHz) and redshift.
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 13))

datasets_fig6 = [
    (axes[0], ds_lna,   corr_discone_h, 'Discone + ZKL-2+ LNA'),
    (axes[1], ds_nolna, corr_discone_h, 'Discone, no LNA'),
    (axes[2], ds_yagi,  corr_yagi_h,    'Yagi + CobraX LNA'),
]

for ax, ds, corr_h, title in datasets_fig6:
    if ds.get('fft_h') is None: continue
    spec_h = ds['fft_h']
    f_h    = freq_hires
    mask_h = (f_h >= RHINO_LO) & (f_h <= RHINO_HI)
    f_band = f_h[mask_h]
    spec_band = spec_h[mask_h] + corr_h  # S11 corrected

    ax.plot(f_band, spec_band, lw=0.4, color=COL['fft'], alpha=0.85,
            label='Hi-res FFT (%.3f kHz/bin) + S11 corr' % DF_HIRES_KHZ)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO, RHINO_HI)
    ax.set_title(title)
    ax.legend(fontsize=8)

    # Redshift axis
    ax2 = ax.twiny()
    f_ticks = np.linspace(RHINO_LO, RHINO_HI, 6)
    ax2.set_xlim(RHINO_LO, RHINO_HI)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)

fig.suptitle('Fig 6 — Hi-res Spectrum at %.3f kHz/bin | N=%d | S11 Corrected' %
             (DF_HIRES_KHZ, N_FFT_HIRES), fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig6_hires_spectrum.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 6 saved')

ValueError: operands could not be broadcast together with shapes (5926,) (5927,) 

In [19]:
# ================================================================
# CELL 12 — Fig 7: Hi-res waterfall at 4.219 kHz/bin
# ================================================================

def make_hires_wf(ds, label, fname):
    wf = ds.get('wf_hires')
    if wf is None: print('  SKIP', label); return
    n_fr, n_bins = wf.shape
    f_hr = np.linspace(RHINO_LO, RHINO_HI, n_bins)
    flat = wf[wf != 0].ravel()
    vmin = float(np.percentile(flat,  5)) if len(flat) else 20
    vmax = float(np.percentile(flat, 95)) if len(flat) else 60

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    im = axes[0].imshow(wf, aspect='auto', origin='upper',
                        extent=[f_hr[0], f_hr[-1], n_fr*10, 0],
                        cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0].set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel('Time (s)')
    axes[0].set_title('Hi-res FFT Waterfall | %.3f kHz/bin | %d frames' %
                      (DF_HIRES_KHZ, n_fr))
    plt.colorbar(im, ax=axes[0],
                 label='dB (%.0f-%.0f dB)' % (vmin, vmax))

    axes[1].plot(f_hr, np.mean(wf, axis=0), color=COL['fft'],
                 lw=0.6, alpha=0.85, label='Mean')
    axes[1].plot(f_hr, np.max(wf, axis=0), color='red',
                 lw=0.6, alpha=0.6, label='Max-hold')
    axes[1].set_xlabel('Frequency (MHz)')
    axes[1].set_ylabel('Power (dB, arb.)')
    axes[1].set_xlim(RHINO_LO, RHINO_HI)
    axes[1].set_title('Mean and Max-hold | %d frames' % n_fr)
    axes[1].legend()

    fig.suptitle('Fig 7 — Hi-res Waterfall | 60-85 MHz | %s | v7 normalisation' % label)
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('  Saved:', os.path.basename(fname))

print('Generating Fig 7 hi-res waterfalls...')
for ds, tag, lbl in [
    (ds_lna,   'lna',   'Discone + ZKL-2+ LNA'),
    (ds_nolna, 'nolna', 'Discone, no LNA'),
    (ds_yagi,  'yagi',  'Yagi + CobraX LNA'),
]:
    make_hires_wf(ds, lbl, OUT + '/fig7_hires_waterfall_%s.png' % tag)

print('PASS Cell 12 — Fig 7 complete')

Generating Fig 7 hi-res waterfalls...
  Saved: fig7_hires_waterfall_lna.png
  Saved: fig7_hires_waterfall_nolna.png
  Saved: fig7_hires_waterfall_yagi.png
PASS Cell 12 — Fig 7 complete


In [20]:
# ================================================================
# CELL 13 — Fig 8: Radiometer equation compliance
# All four sky/receiver datasets on one log-log plot.
# Reference sub-band: 60-75 MHz (RFI-quiet).
# ================================================================

def radiometer_curve(snap_Ns, snaps, freq_axis, ref_lo, ref_hi):
    """Compute spectral std dev in ref band vs N."""
    if not snap_Ns: return [], []
    ref_mask = (freq_axis >= ref_lo) & (freq_axis <= ref_hi)
    Ns_out, stds_out = [], []
    for N_val, arr in zip(snap_Ns, snaps):
        if arr.size == len(freq_axis) and ref_mask.sum() > 0:
            std = float(np.std(arr[ref_mask]))
            if np.isfinite(std) and std > 0:
                Ns_out.append(N_val); stds_out.append(std)
    return Ns_out, stds_out

print('Computing radiometer curves...')

curves = []
for ds, col, lbl, integ_freq_key in [
    (ds_lna,   COL['lna'],  'Discone+LNA',   'integ_freq'),
    (ds_nolna, COL['nolna'],'Discone No-LNA','integ_freq'),
    (ds_yagi,  COL['yagi'], 'Yagi+CobraX',   'integ_freq'),
    (ds_load,  COL['load'], 'Load+LNA (3h)', 'integ_freq'),
]:
    if not ds['snap_Ns']:
        print('  SKIP', lbl, '— no snaps'); continue

    # Use stored freq axis if available, else use computed one
    f_ax = ds.get(integ_freq_key)
    if f_ax is None:
        f_ax = freq_hires
    elif f_ax.max() < 10:
        f_ax = f_ax * 1e6
    elif f_ax.max() > 1e6:
        f_ax = f_ax / 1e6

    Ns, stds = radiometer_curve(
        ds['snap_Ns'], ds['snaps'], f_ax, REF_LO, REF_HI)
    if not Ns: print('  SKIP', lbl, '— no valid stds'); continue

    # Sort by N (defensive: guarantees endpoints are true min/max N)
    order    = np.argsort(Ns)
    Ns_arr   = np.array(Ns,   dtype=float)[order]
    stds_arr = np.array(stds, dtype=float)[order]

    # Compliance ratio: R = sigma(N_min)*sqrt(N_min) / [sigma(N_max)*sqrt(N_max)]
    # R = 1 -> ideal 1/sqrt(N); R < 1 -> noise floored out (real structure / systematics)
    R = (stds_arr[0]*np.sqrt(Ns_arr[0])) / (stds_arr[-1]*np.sqrt(Ns_arr[-1]))
    curves.append((Ns_arr, stds_arr, col, lbl, R))
    print('  %s: N=%d to %d | R=%.3f' % (lbl, int(Ns_arr[0]), int(Ns_arr[-1]), R))

fig, ax = plt.subplots(figsize=(10, 6))
if curves:
    N_all = np.concatenate([c[0] for c in curves])
    N_id  = np.geomspace(N_all.min(), N_all.max(), 200)
    # Ideal 1/sqrt(N) anchored to first dataset
    scale = curves[0][1][0] * np.sqrt(curves[0][0][0])
    ax.plot(N_id, scale/np.sqrt(N_id), 'k--', lw=1.5, alpha=0.5,
            label='Ideal 1/√N (radiometer equation)')
    for Ns, stds, col, lbl, R in curves:
        ax.scatter(Ns, stds, s=15, color=col, zorder=5)
        ax.plot(Ns, stds, color=col, lw=1.0,
                label='%s  R=%.2f' % (lbl, R))
        ax.annotate('ratio=%.2f' % R, xy=(Ns[-1], stds[-1]),
                    xytext=(5, 0), textcoords='offset points',
                    fontsize=8, color=col)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('N (averaged spectra)')
ax.set_ylabel('Spectral std dev in %.0f-%.0f MHz (dB)' % (REF_LO, REF_HI))
ax.set_title('Fig 8 — Radiometer Equation | All v2 Datasets\n'
             'Reference sub-band: %.0f-%.0f MHz' % (REF_LO, REF_HI))
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT + '/fig8_radiometer_equation.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 8 saved')

Computing radiometer curves...
  Discone+LNA: N=12 to 350 | R=0.247
  Discone No-LNA: N=12 to 350 | R=0.258
  Yagi+CobraX: N=12 to 350 | R=0.261
  Load+LNA (3h): N=100 to 19800 | R=0.822
PASS Fig 8 saved


In [ ]:
# ================================================================
# CELL 14 — S11 correction diagnostic figure
# Shows the S11 curves, mismatch efficiency, and the correction
# applied to the discone and Yagi spectra.
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

f_plot = np.linspace(RHINO_LO, RHINO_HI, 500)

for ax_s11, ax_eta, freq_s11, s11_db_raw, label, col in [
    (axes[0,0], axes[0,1],
     freq_s11_discone if HAS_DISCONE_S11 else None,
     s11_db_discone   if HAS_DISCONE_S11 else None,
     'Discone', COL['lna']),
    (axes[1,0], axes[1,1],
     freq_s11_yagi if HAS_YAGI_S11 else None,
     s11_db_yagi   if HAS_YAGI_S11 else None,
     'Yagi', COL['yagi']),
]:
    if freq_s11 is None:
        ax_s11.text(0.5, 0.5, 'S11 not available', ha='center',
                   transform=ax_s11.transAxes)
        ax_eta.text(0.5, 0.5, 'S11 not available', ha='center',
                   transform=ax_eta.transAxes)
        continue

    spline  = UnivariateSpline(freq_s11, s11_db_raw, k=3, s=0.5)
    s11_fit = spline(f_plot)
    gamma   = 10**(s11_fit / 20.0)
    eta     = np.clip(1 - gamma**2, 0.01, 1.0)
    corr_db = -10 * np.log10(eta)

    # S11 plot
    mask_band = (freq_s11 >= RHINO_LO) & (freq_s11 <= RHINO_HI)
    ax_s11.scatter(freq_s11[mask_band], s11_db_raw[mask_band],
                   s=8, color=col, alpha=0.5, label='Measured points')
    ax_s11.plot(f_plot, s11_fit, color=col, lw=1.5,
                label='Cubic spline fit')
    ax_s11.set_xlabel('Frequency (MHz)')
    ax_s11.set_ylabel('|S11| (dB)')
    ax_s11.set_title('%s S11 | %d points in 60-85 MHz' %
                     (label, mask_band.sum()))
    ax_s11.set_xlim(RHINO_LO, RHINO_HI)
    ax_s11.legend(fontsize=8)

    # Mismatch efficiency and correction
    ax_eta.plot(f_plot, eta, color=col, lw=1.5,
                label='η = 1 - |Γ|²  (power received)')
    ax_eta2 = ax_eta.twinx()
    ax_eta2.plot(f_plot, corr_db, color='grey', lw=1.0,
                 ls='--', label='Correction (dB)')
    ax_eta2.set_ylabel('Correction added (dB)', color='grey')
    ax_eta.set_xlabel('Frequency (MHz)')
    ax_eta.set_ylabel('Mismatch efficiency η')
    ax_eta.set_title('%s Mismatch Efficiency | mean η=%.3f' %
                     (label, eta.mean()))
    ax_eta.set_xlim(RHINO_LO, RHINO_HI)
    ax_eta.set_ylim(0, 1.05)
    ax_eta.legend(fontsize=8, loc='upper left')
    ax_eta2.legend(fontsize=8, loc='upper right')

fig.suptitle('S11 Correction Diagnostic | Discone and Yagi Antennas\n'
             'Jordan\'s method: cubic spline fit to |S11| in dB',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig_s11_correction_diagnostic.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Cell 14 — S11 diagnostic saved')

In [ ]:
# ================================================================
# CELL 15 — Session summary: list all saved figures
# ================================================================
print('=' * 60)
print('THESIS FIGURES SUMMARY')
print('=' * 60)
print('Output directory:', OUT)
print()
all_figs = sorted(glob.glob(OUT + '/*.png'))
for fp in all_figs:
    size_kb = os.path.getsize(fp) / 1024
    print('  %-55s %6.0f KB' % (os.path.basename(fp), size_kb))
print()
print('Total figures:', len(all_figs))
print()
print('S11 corrections applied:')
print('  Discone LNA    :', 'YES (%.0f pts, mean η=%.3f)' %
      (249, eta_discone_c.mean()) if HAS_DISCONE_S11 else 'NO')
print('  Discone No-LNA :', 'YES (same S11)' if HAS_DISCONE_S11 else 'NO')
print('  Yagi           :', 'YES (%.0f pts, mean η=%.3f)' %
      (332, eta_yagi_c.mean()) if HAS_YAGI_S11 else 'NO')
print('  Load           : N/A (not an antenna)')
print()
print('PASS Cell 15 — complete')